<a href="https://colab.research.google.com/github/madanjha/Machine-Learning/blob/main/GenAi_LamaCpp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import kagglehub
import os
import matplotlib.pyplot as plt
from sklearn.utils import resample
import re
from tensorflow.keras.preprocessing.text import Tokenizer
import warnings
warnings.filterwarnings('ignore')

In [2]:
#Step -1 load the data
print(f"Amazon fine food reviews data set")
path = kagglehub.dataset_download("snap/amazon-fine-food-reviews")
csv_file = os.path.join(path, "Reviews.csv")
df = pd.read_csv(csv_file)

Amazon fine food reviews data set


100%|██████████| 242M/242M [00:01<00:00, 198MB/s]

Extracting files...


In [3]:
df.head(2) # 1-2-3-4-5. review-
#df['Score'].value_counts()

,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
0,1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,5,1303862400,Good Quality Dog Food,I have bought several of the Vitality canned d...
1,2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0,1,1346976000,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...


In [4]:
!pip install langchain langchain-community pypdf docarray sentence-transformers huggingface_hub llama-cpp-python -q
!apt-get update && apt-get install -y git cmake build-essential -q

# Install llama.cpp
!git clone https://github.com/ggerganov/llama.cpp
%cd llama.cpp
!make -j$(nproc)
%cd ..

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.8/49.8 MB 11.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 305.5/305.5 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.8/302.8 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.2/45.2 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 107.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 81.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 42.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━

In [5]:
from huggingface_hub import hf_hub_download

model_path = hf_hub_download(repo_id="bartowski/Llama-3.2-3B-Instruct-GGUF", filename="Llama-3.2-3B-Instruct-Q4_K_M.gguf")

# bartowski/Llama-3.2-3B-Instruct-GGUF

# # Llama-3.2-3B-Instruct-Q4_K_M.gguf

Llama-3.2-3B-Instruct-Q4_K_M.gguf:   0%|          | 0.00/2.02G [00:00<?, ?B/s]

In [6]:
from google.colab import files
uploaded = files.upload()
pdf_path = list(uploaded.keys())[0]

Saving IITK_DA_GenAI new.pdf to IITK_DA_GenAI new.pdf


In [7]:
import torch
from langchain_community.llms import LlamaCpp

In [10]:
llm = LlamaCpp(
    model_path=model_path,
    n_ctx = 2048, #context window size 2048 tokens at once prompt +gen
    n_gpu_layers = 33 if torch.cuda.is_available() else 0,
    temperature = 0.7, #0.0 determnistm 0.7 random
    verbose=False #quite
)

llama_context: n_batch is less than GGML_KQ_MASK_PAD - increasing to 64
llama_context: n_ctx_per_seq (2048) < n_ctx_train (131072) -- the full capacity of the model will not be utilized
llama_kv_cache_unified: LLAMA_SET_ROWS=0, using old ggml_cpy() method for backwards compatibility


In [11]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import DocArrayInMemorySearch
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [12]:
loader = PyPDFLoader(pdf_path)
pages = loader.load_and_split()

In [13]:
pages

[Document(metadata={'producer': 'Adobe PDF library 17.00', 'creator': 'Adobe Illustrator 27.1 (Windows)', 'creationdate': '2024-05-14T18:40:03+06:30', 'moddate': '2024-05-14T18:40:03+05:30', 'title': 'IITK_DA_GenAI', 'trapped': '/False', 'source': 'IITK_DA_GenAI new.pdf', 'total_pages': 35, 'page': 0, 'page_label': '1'}, page_content='Data Analytics &\nGenerative AI\nPROFESSIONAL CERTIFICATE COURSE IN'),
 Document(metadata={'producer': 'Adobe PDF library 17.00', 'creator': 'Adobe Illustrator 27.1 (Windows)', 'creationdate': '2024-05-14T18:40:03+06:30', 'moddate': '2024-05-14T18:40:03+05:30', 'title': 'IITK_DA_GenAI', 'trapped': '/False', 'source': 'IITK_DA_GenAI new.pdf', 'total_pages': 35, 'page': 1, 'page_label': '2'}, page_content='About the Program         03\nAbout E&ICT Academy, IIT Kanpur       04\nAbout Simplilearm       04\nKey Features of the Program         05\nEligibility Criteria         06\nData Analytics & Generative AI Industry Trends      08\nWho is this Program Ideal 

In [14]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

/tmp/ipython-input-14-3409896792.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [15]:
#vector store
vectorstore = DocArrayInMemorySearch.from_documents(pages, embedding=embeddings)
retriever = vectorstore.as_retriever()

#origibal text, embd, meta data

# my name is rudra - [0.12, 0.34,0.98 ....] [fgile_nbame] [size]

In [16]:
retriever

VectorStoreRetriever(tags=['DocArrayInMemorySearch'], vectorstore=<langchain_community.vectorstores.docarray.in_memory.DocArrayInMemorySearch object at 0x7e5190bf9710>, search_kwargs={})

In [18]:
#prompt

#basiz 0shot
template = """Use the following context to answer the question:
{context}

Question : {question}

Answer: """
prompt = PromptTemplate.from_template(template)

In [19]:
#RAG Chain - invoke
chain = (
    {"context":retriever, "question":RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser() # extrat response llm  -> give you in a proper format  indenraiom pace !
)

In [20]:
question = "how to make pizza" #control res
response = chain.invoke(question)

In [21]:
print(f"Answer:{response}")

Answer:1. Start by preheating your oven to 450°F (230°C). Make sure it's hot before you start making the pizza.

2. While the oven is heating up, let's prepare our ingredients for the pizza. You will need some flour, yeast, salt, sugar, olive oil, and any toppings you want on your pizza!

3. Now that we have all of our ingredients ready, let’s make the crust of our pizza. To do this, combine 2 cups of warm water (around 100°F), 1 tablespoon of sugar, and 1 teaspoon of active dry yeast in a large mixing bowl. Let the mixture sit for about 10 minutes until it becomes frothy and bubbly.

4. Once the dough has risen to your liking, punch it down and divide it into two equal portions. Roll out each portion of the dough into a thin circle, approximately 12 inches in diameter. You can either shape the dough by hand or use a pizza stone or baking sheet to help you shape the dough.

5. Now that we have our dough shaped into circles, let’s top our pizza with your desired toppings. Some popular t

In [22]:
import time

st = time.time()
question = "What is the main topic of the PDF? can you tell me top 2 important points from the document?"
response = chain.invoke(question)
print("Answer:{response}")
ed = time.time()
print(f"Inference is at :{(ed-st):.3f} Seconds")

Answer:{response}
Inference is at :211.568 Seconds
